In [14]:
import torch
import torch.utils

# 1. Vá lỗi thiếu int1-int7 và uint1-uint7
for i in range(1, 8):
    int_attr = f"int{i}"
    uint_attr = f"uint{i}"
    if not hasattr(torch, int_attr):
        setattr(torch, int_attr, torch.int8)
    if not hasattr(torch, uint_attr):
        setattr(torch, uint_attr, torch.uint8)

# 2. Vá lỗi thiếu register_constant trong torch.utils._pytree
if not hasattr(torch.utils, "_pytree"):
    import torch.utils._pytree
if not hasattr(torch.utils._pytree, "register_constant"):
    torch.utils._pytree.register_constant = lambda cls: cls

# 3. Tắt TorchAO trong transformers để tránh các lỗi xung đột gián tiếp khác
import transformers
transformers.utils.import_utils.is_torchao_available = lambda *args, **kwargs: False
transformers.utils.is_torchao_available = lambda *args, **kwargs: False
if hasattr(transformers.utils.import_utils, "_torchao_available"):
    transformers.utils.import_utils._torchao_available = False

print("✅ Đã hoàn tất vá lỗi tương thích hệ thống hoàn chỉnh!")

✅ Đã hoàn tất vá lỗi tương thích hệ thống hoàn chỉnh!


In [15]:
import os
import yaml
import pandas as pd
import json

# Chuyển Jupyter về thư mục huấn luyện llm-training
os.chdir("/mnt/hungpv/car_bench_sft/car_bench_notebook/llm-training")

# XÓA HOÀN TOÀN các biến môi trường DDP để tránh kích hoạt distributed training
for key in ["WORLD_SIZE", "RANK", "LOCAL_RANK", "MASTER_ADDR", "MASTER_PORT"]:
    if key in os.environ:
        del os.environ[key]

# Đọc cấu hình huấn luyện
CONFIG_PATH = "ddp_config.yml"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Đã nạp cấu hình thành công ở chế độ Single-GPU!")
print(f"Mô hình đích: {config['model']['name']}")
print(f"File dữ liệu mẫu: {config['datasets']['sample']['path']}")

Đã nạp cấu hình thành công ở chế độ Single-GPU!
Mô hình đích: unsloth/Qwen3.5-0.8B
File dữ liệu mẫu: data/sample.jsonl


In [16]:
!export CUDA_VISIBLE_DEVICE=1

In [17]:
from transformers import AutoTokenizer

# Tải tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    config["model"]["name"], 
    trust_remote_code=True
)

# Đọc thử file dữ liệu
dataset_path = config["datasets"]["sample"]["path"]
print(f"Đọc dữ liệu từ: {dataset_path}")
df = pd.read_json(dataset_path, lines=True).head(5)

# Chạy thử chuyển đổi
print("Chạy thử áp dụng Chat Template và Tokenize trên 5 dòng đầu:")
for i, row in enumerate(df.itertuples(index=False), 1):
    try:
        # Lấy tools nếu có
        tools = getattr(row, "tools", None) or None
        convs = getattr(row, "conversations")
        
        # Áp dụng template
        text = tokenizer.apply_chat_template(
            convs,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
            tools=tools,
        )
        # Tokenize thử
        enc = tokenizer(text, truncation=True, max_length=1024)
        print(f"  [Dòng {i}] Độ dài tokens: {len(enc['input_ids'])} -> OK")
        
    except Exception as e:
        print(f"  ❌ [Dòng {i}] BỊ LỖI: {e}")

Loading tokenizer...
Đọc dữ liệu từ: data/sample.jsonl
Chạy thử áp dụng Chat Template và Tokenize trên 5 dòng đầu:
  [Dòng 1] Độ dài tokens: 366 -> OK
  [Dòng 2] Độ dài tokens: 365 -> OK
  [Dòng 3] Độ dài tokens: 362 -> OK
  [Dòng 4] Độ dài tokens: 366 -> OK
  [Dòng 5] Độ dài tokens: 365 -> OK


In [18]:
from unsloth import FastLanguageModel
import torch

print("Loading model in 4-bit for smoke testing...")
model, processor = FastLanguageModel.from_pretrained(
    model_name=config["model"]["name"],
    max_seq_length=2048, # Giới hạn độ dài ngắn để chạy test nhanh
    dtype="bfloat16" if torch.cuda.is_bf16_supported() else "float16",
    load_in_4bit=True,   # Ép 4-bit để test nhanh trên 1 GPU
)
tokenizer = processor.tokenizer

# Áp dụng cấu hình LoRA PEFT
model = FastLanguageModel.get_peft_model(
    model,
    r=config["lora"]["r"],
    target_modules=config["lora"]["target_modules"],
    lora_alpha=config["lora"]["alpha"],
    lora_dropout=0.0,    # Unsloth tối ưu tốt nhất ở 0.0
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("✅ Nạp mô hình & LoRA thành công!")

Loading model in 4-bit for smoke testing...
==((====))==  Unsloth 2026.6.9: Fast Qwen3_5 patching. Transformers: 5.12.1.
   \\   /|    NVIDIA A100-PCIE-40GB. Num GPUs = 2. Max memory: 39.564 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 8.0. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|█████████████████████████████████████████████████████████████| 473/473 [00:00<00:00, 1046.21it/s]


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
✅ Nạp mô hình & LoRA thành công!


In [12]:
from datasets import Dataset

input_ids_list = []
attention_mask_list = []

for row in df.itertuples(index=False):
    tools = getattr(row, "tools", None) or None
    convs = getattr(row, "conversations")
    text = tokenizer.apply_chat_template(
        convs, tokenize=False, add_generation_prompt=False, enable_thinking=False, tools=tools
    )
    enc = tokenizer(text, truncation=True, max_length=2048)
    input_ids_list.append(enc["input_ids"])
    attention_mask_list.append(enc["attention_mask"])

dummy_dataset = Dataset.from_dict({
    "input_ids": input_ids_list,
    "attention_mask": attention_mask_list
})
print(f"Tạo dummy dataset thành công: {len(dummy_dataset)} dòng.")

Tạo dummy dataset thành công: 5 dòng.


In [13]:
from trl import SFTTrainer, SFTConfig
from unsloth import train_on_responses_only

# Cấu hình huấn luyện dummy
training_args = SFTConfig(
    output_dir="/tmp/dummy_output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_length=2048,
    packing=False,
    dataset_text_field=None,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dummy_dataset,
    args=training_args,
)

# Áp dụng bộ lọc Masking Loss
response_cfg = config["training"].get("response_format", {})
instruction_part = response_cfg.get("instruction_part", "<|im_start|>user\n")
response_part = response_cfg.get("response_part", "<|im_start|>assistant\n<think>\n\n</think>\n\n")

trainer = train_on_responses_only(
    trainer,
    instruction_part=instruction_part,
    response_part=response_part,
)

# Giải mã thử mẫu đầu tiên để xem phần nào được tính Loss
# Những phần bị thay bằng khoảng trắng/không có nhãn là những phần bị bỏ qua (labels = -100)
example_answer = tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]
).replace(tokenizer.pad_token, " ")

print("=== PHẦN MÔ HÌNH SẼ TÍNH LOSS (Không bị mask) ===")
print(example_answer)

ValueError: Error initializing torch.distributed using env:// rendezvous: environment variable MASTER_ADDR expected, but not set